In [8]:
import pandas as pd
import numpy as np
import pickle
from datetime import datetime

DATA_PATH = '../data'

# Load what we need
full_dataset = pd.read_parquet(f'{DATA_PATH}/full_dataset.parquet')
movies = pd.read_csv(f'{DATA_PATH}/movies.csv')
ratings = pd.read_csv(f'{DATA_PATH}/ratings.csv')

# Load propensity model
with open(f'{DATA_PATH}/xgb_model_v2.pkl', 'rb') as f:
    propensity_model = pickle.load(f)

print("Loaded successfully")

Loaded successfully


In [9]:
def time_relevance_score(user_peak_hour, user_peak_season,
                         movie_peak_hour, movie_peak_season,
                         user_rating_count):
    
    # Pattern strength - more ratings = more reliable pattern
    pattern_strength = min(user_rating_count / 200, 1.0)
    
    # Hour match - within 2 hours = full score
    hour_match = 1.0 if abs(user_peak_hour - movie_peak_hour) <= 2 else 0.8
    
    # Season match
    season_match = 1.0 if user_peak_season == movie_peak_season else 0.9
    
    # Combined nudge - stays between 0.81 and 1.0
    raw_score = hour_match * season_match
    nudge = 1.0 + (raw_score - 1.0) * pattern_strength
    
    return nudge

# Test it
print("Active user, perfect match:", time_relevance_score(18, 'winter', 18, 'winter', 500))
print("Active user, no match:", time_relevance_score(18, 'winter', 22, 'summer', 500))
print("New user, perfect match:", time_relevance_score(18, 'winter', 18, 'winter', 20))
print("New user, no match:", time_relevance_score(18, 'winter', 22, 'summer', 20))

Active user, perfect match: 1.0
Active user, no match: 0.7200000000000001
New user, perfect match: 1.0
New user, no match: 0.972


In [10]:
with open(f'{DATA_PATH}/svd_model.pkl', 'rb') as f:
    svd_model = pickle.load(f)

print("SVD model loaded!")

SVD model loaded!


In [12]:
def get_recommendations(user_id, propensity_model, svd_model, 
                        full_dataset, movies, n=10):
    
    # Get user features
    user_data = full_dataset[full_dataset['userId'] == user_id].iloc[0]
    user_peak_hour = user_data['user_peak_hour']
    user_peak_season = user_data['user_peak_season']
    user_rating_count = user_data['user_rating_count']
    
    # Movies user already watched
    watched_movies = full_dataset[
        (full_dataset['userId'] == user_id) & 
        (full_dataset['watched'] == 1)
    ]['movieId'].tolist()
    
    # Candidate movies — everything not watched
    candidates = full_dataset[
        ~full_dataset['movieId'].isin(watched_movies)
    ].drop_duplicates('movieId')
    
    print(f"Candidate movies: {len(candidates)}")
    return candidates

# Test with user 1
candidates = get_recommendations(1, propensity_model, svd_model, 
                                  full_dataset, movies)

Candidate movies: 58977


In [18]:
def get_recommendations(user_id, propensity_model, svd_model,
                        full_dataset, movies, n=10):

    # Get user features
    user_data = full_dataset[full_dataset['userId'] == user_id].iloc[0]
    user_peak_hour = user_data['user_peak_hour']
    user_peak_season = user_data['user_peak_season']
    user_rating_count = user_data['user_rating_count']

    # Movies user already watched
    watched_movies = full_dataset[
        (full_dataset['userId'] == user_id) &
        (full_dataset['watched'] == 1)
    ]['movieId'].tolist()

    # Candidate movies — everything not watched
    candidates = full_dataset[
        ~full_dataset['movieId'].isin(watched_movies)
    ].drop_duplicates('movieId').copy()

    # ← NEW: filter out obscure movies
    candidates = candidates[candidates['movie_rating_count'] >= 500]

    print(f"Candidate movies after popularity filter: {len(candidates)}")

    # --- Signal 1: Propensity Score ---
    drop_cols = ['userId', 'movieId', 'watched', 'user_peak_season', 'movie_peak_season',
                 'movie_rating_count', 'user_rating_count']
    X_candidates = candidates.drop(columns=drop_cols).fillna(0)
    candidates['propensity_score'] = propensity_model.predict_proba(X_candidates)[:, 1]

    # --- Signal 2: SVD Score ---
    user_idx = svd_model.train_set.uid_map.get(str(user_id))
    if user_idx is not None:
        svd_scores = svd_model.score(user_idx)
        idx_to_item = {v: k for k, v in svd_model.train_set.iid_map.items()}
        candidates['svd_score'] = candidates['movieId'].astype(str).map(
            {idx_to_item[i]: svd_scores[i] for i in range(len(svd_scores))}
        ).fillna(svd_scores.mean())
        candidates['svd_score'] = (candidates['svd_score'] - svd_scores.min()) / (svd_scores.max() - svd_scores.min())
    else:
        candidates['svd_score'] = 0.5

    # --- Signal 3: Time Relevance Score ---
    candidates['time_score'] = candidates.apply(
        lambda row: time_relevance_score(
            user_peak_hour, user_peak_season,
            row['movie_peak_hour'], row['movie_peak_season'],
            user_rating_count
        ), axis=1
    )

    # --- Final Ranking ---
    candidates['final_score'] = (
        candidates['propensity_score'] *
        candidates['svd_score'] *
        candidates['time_score']
    )

    # Get top N
    top_n = candidates.nlargest(n, 'final_score')[['movieId', 'propensity_score', 'svd_score', 'time_score', 'final_score']]
    movies['movieId'] = movies['movieId'].astype(str)
    top_n['movieId'] = top_n['movieId'].astype(str)
    top_n = top_n.merge(movies, on='movieId')

    return top_n[['title', 'genres', 'propensity_score', 'svd_score', 'time_score', 'final_score']]

# Test with user 1
results = get_recommendations(1, propensity_model, svd_model, full_dataset, movies)
print(results)

Candidate movies after popularity filter: 5333
                                               title  \
0                   Shawshank Redemption, The (1994)   
1        Seven Samurai (Shichinin no samurai) (1954)   
2                                     Yojimbo (1961)   
3                              Mulholland Dr. (1999)   
4                         Usual Suspects, The (1995)   
5  Lives of Others, The (Das leben der Anderen) (...   
6  Scenes From a Marriage (Scener ur ett äktenska...   
7                                Donnie Darko (2001)   
8                                  Fight Club (1999)   
9                              Third Man, The (1949)   

                          genres  propensity_score  svd_score  time_score  \
0                    Crime|Drama          0.965558   0.950867       0.902   
1         Action|Adventure|Drama          0.961778   0.950665       0.902   
2               Action|Adventure          0.956142   0.952618       0.902   
3          Drama|Mystery|Rom

In [15]:
# Convert both to same type
ratings_user1 = ratings[ratings['userId'] == 1].copy()
ratings_user1['movieId'] = ratings_user1['movieId'].astype(str)
movies['movieId'] = movies['movieId'].astype(str)

ratings_user1 = ratings_user1.merge(movies, on='movieId')
ratings_user1 = ratings_user1.sort_values('rating', ascending=False)

print("User 1 top rated movies:")
print(ratings_user1[['title', 'genres', 'rating']].head(10))

print("\nOur recommendations:")
print(results[['title', 'genres', 'final_score']])

User 1 top rated movies:
                                                title  \
0                                 Pulp Fiction (1994)   
18  Saragossa Manuscript, The (Rekopis znaleziony ...   
57                                       Dolls (2002)   
56                              Dolce Vita, La (1960)   
48       Eternal Sunshine of the Spotless Mind (2004)   
41                         Lost in Translation (2003)   
37                City of God (Cidade de Deus) (2002)   
33                            Teddy Bear (Mis) (1981)   
26                      Night, The (Notte, La) (1960)   
24      In the Mood For Love (Fa yeung nin wa) (2000)   

                                   genres  rating  
0             Comedy|Crime|Drama|Thriller     5.0  
18                Adventure|Drama|Mystery     5.0  
57                          Drama|Romance     5.0  
56                                  Drama     5.0  
48                   Drama|Romance|Sci-Fi     5.0  
41                   Comedy|Drama|R

In [16]:
# Check One Froggy Evening's features
one_froggy = full_dataset[full_dataset['movieId'] == 30701].iloc[0]
print(one_froggy[['movie_rating_count', 'movie_avg_rating', 'movie_rating_std', 
                   'movie_peak_hour', 'movie_peak_season']])

movie_rating_count         127
movie_avg_rating      3.688976
movie_rating_std      1.112571
movie_peak_hour              3
movie_peak_season         fall
Name: 100868, dtype: object
